# SKENARIO 1 — Pilih Teacher CoT Terbaik

Bandingkan 3 generator CoT: **gemma**, **deepseek**, **ernie**.

Metrik = seberapa sedikit kandidat HILANG saat difilter judge (`cot_*` -> `correct_*`).
Makin sedikit berkurang = teacher makin sering benar = makin bagus.

```
retention% = correct / candidates * 100   (makin TINGGI makin bagus)
reduction% = 100 - retention%             (makin RENDAH makin bagus)
```

Tanpa GPU. Cukup baca file `cot_*`/`correct_*` hasil tiap pipeline teacher.
Pemenang -> pakai `sft/` dari teacher itu buat training (skenario lain).

## 0. Setup (clone/samain repo)

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = '/kaggle/working/FP_NLP'
URL  = 'https://github.com/henray404/FP_NLP.git'
if os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'fetch','-q','origin','main'], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','-q',URL,REPO], check=True)
sys.path.insert(0, REPO); os.chdir(REPO)
print('repo ready:', REPO)

## 1. Tunjuk file per teacher

Auto-cari `cot_*`/`correct_*` di `/kaggle/input` (Add dataset hasil CoT tiap teacher) dan di repo `data/cot/`.
Tag teacher diambil dari nama file/folder. Kalau salah deteksi, isi glob manual.

In [ ]:
# Per teacher: (glob kandidat, glob correct). ** = rekursif. Tambah/hapus sesuai run yang ada.
TEACHERS = {
    'gemma': {
        'candidates': ['/kaggle/input/**/cot_*Qwen*.jsonl', 'data/cot/**/cot_*Qwen*.jsonl'],
        'correct':    ['/kaggle/input/**/correct_*Qwen*.jsonl', 'data/cot/**/correct_*Qwen*.jsonl'],
    },
    'deepseek': {
        'candidates': ['/kaggle/input/**/cot_*DeepSeek*.jsonl', 'data/cot/**/cot_*DeepSeek*.jsonl'],
        'correct':    ['/kaggle/input/**/correct_*DeepSeek*.jsonl', 'data/cot/**/correct_*DeepSeek*.jsonl'],
    },
    'ernie': {
        'candidates': ['/kaggle/input/**/cot_*ernie*.jsonl', '/kaggle/input/**/cot_*ERNIE*.jsonl', 'data/cot/**/cot_*ernie*.jsonl'],
        'correct':    ['/kaggle/input/**/correct_*ernie*.jsonl', '/kaggle/input/**/correct_*ERNIE*.jsonl', 'data/cot/**/correct_*ernie*.jsonl'],
    },
}
print('teachers:', list(TEACHERS))

## 2. Komparasi + pemenang (tabel)

In [ ]:
from src.cot_synthesis.compare_teachers import compare, render_table

result = compare(TEACHERS)
print(render_table(result))
print()
print('WINNER:', result['winner'], '-> pakai sft/ teacher ini buat training (S2/S3)')

import json
Path('data/eval').mkdir(parents=True, exist_ok=True)
Path('data/eval/s1_teachers.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print('summary -> data/eval/s1_teachers.json')